# Bank Customer Churn Prediction (Neural Network)

**Dataset:** `Churn_Modelling.csv`  
**Goal:** Predict whether a customer will leave the bank (`Exited` = 1) using a neural network, then improve the model with more neurons and Dropout.

## 1. Install requirements (only needed once, e.g. in Colab)

In [ ]:
# Uncomment the next line if a library is missing
# %pip install -q pandas numpy scikit-learn tensorflow

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

# Same results every run
np.random.seed(42)
tf.random.set_seed(42)

## 3. Load the dataset

In [ ]:
import os

def find_file(name):
    """Look for a dataset in the usual places; in Google Colab, ask for an upload."""
    folders = [".", "data", "/content", "/content/drive/MyDrive", "/mnt/data",
               os.path.expanduser("~/Downloads"), os.path.expanduser("~/Desktop")]
    for folder in folders:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            print(f"Found dataset: {path}")
            return path
    try:
        from google.colab import files   # only exists in Google Colab
        print(f"'{name}' not found. Please upload it now:")
        files.upload()
        if os.path.exists(name):
            return name
    except ImportError:
        pass
    raise FileNotFoundError(
        f"Could not find '{name}'. Put it in the same folder as this notebook and re-run."
    )

In [ ]:
df = pd.read_csv(find_file("Churn_Modelling.csv"))
print("Shape:", df.shape)
df.head()

## 4. Preprocessing
Drop ID-like columns, one-hot encode the categorical columns, split and scale.

In [ ]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], errors="ignore")
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)

X = df.drop(columns=["Exited"]).astype("float32")
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
n_features = X_train.shape[1]

print("Features:", n_features)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Helper to train and test a model

In [ ]:
def train_and_test(model):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
    pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))
    return accuracy_score(y_test, pred)

## 6. Basic model

In [ ]:
print("Basic model")
basic = Sequential([
    Input(shape=(n_features,)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid"),
])
basic_acc = train_and_test(basic)

## 7. Improved model
More neurons plus Dropout to reduce overfitting.

In [ ]:
print("Improved model")
improved = Sequential([
    Input(shape=(n_features,)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])
improved_acc = train_and_test(improved)

## 8. Comparison

In [ ]:
print(f"Basic model accuracy    : {basic_acc:.4f}")
print(f"Improved model accuracy : {improved_acc:.4f}")